# Data Exploration — OpenGreenMetric Emission Factor Datasets

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')
print('Data directory:', DATA_DIR)
print('Available files:', os.listdir(DATA_DIR))

## Load Datasets

In [ ]:
with open(os.path.join(DATA_DIR, 'product-category-benchmarks.json')) as f:
    benchmarks = json.load(f)

categories = list(benchmarks['categories'].keys())
print(f'Number of product categories: {len(categories)}')
print('Categories:', categories)

In [ ]:
with open(os.path.join(DATA_DIR, 'defra-conversion-factors.json')) as f:
    defra = json.load(f)

materials = list(defra['materials'].keys())
print(f'Number of DEFRA material factors: {len(materials)}')
for m in materials[:10]:
    info = defra['materials'][m]
    print(f"  {m}: {info['kgCO2ePerKg']:.2f} kgCO2e/kg ({info['category']})")

In [ ]:
with open(os.path.join(DATA_DIR, 'epa-ghg-emission-factors.json')) as f:
    epa = json.load(f)

regions = epa['electricity']['regions']
intl = epa['electricity']['international']
print(f'US grid regions: {len(regions)}')
print(f'International grids: {len(intl)}')
print(f"US average intensity: {epa['electricity']['us_average']['kgCO2ePerKwh']} kgCO2e/kWh")

## Benchmark Statistics

In [ ]:
rows = []
for cat, data in benchmarks['categories'].items():
    rows.append({
        'category': cat,
        'co2e_min': data['co2eKg']['min'],
        'co2e_max': data['co2eKg']['max'],
        'co2e_median': data['co2eKg']['median'],
        'water_min': data['waterLiters']['min'],
        'water_max': data['waterLiters']['max'],
        'water_median': data['waterLiters']['median'],
        'energy_min': data['energyKwh']['min'],
        'energy_max': data['energyKwh']['max'],
        'energy_median': data['energyKwh']['median'],
        'price_median': data['typicalPrice']['median'],
        'lifespan_years': data['typicalLifespan']['years'],
    })

df = pd.DataFrame(rows).set_index('category')
df.describe()

## Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(df['co2e_median'], bins=15, color='#2ecc71', edgecolor='white')
axes[0].set_title('CO\u2082e Median Distribution')
axes[0].set_xlabel('kg CO\u2082e')
axes[0].set_ylabel('Count')

axes[1].hist(df['water_median'], bins=15, color='#3498db', edgecolor='white')
axes[1].set_title('Water Median Distribution')
axes[1].set_xlabel('Liters')
axes[1].set_ylabel('Count')

axes[2].hist(df['energy_median'], bins=15, color='#e67e22', edgecolor='white')
axes[2].set_title('Energy Median Distribution')
axes[2].set_xlabel('kWh')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../figures/distribution_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation Matrix

In [ ]:
numeric_cols = ['co2e_median', 'water_median', 'energy_median', 'price_median', 'lifespan_years']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Environmental Impact Metrics')
plt.tight_layout()
plt.savefig('../figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Findings

- **CO2e, water, and energy** metrics are positively correlated across product categories.
- **Electronics** (laptops, desktops, TVs) dominate the high end of all three distributions.
- **Price** is moderately correlated with environmental impact, but lifespan shows weaker correlation.
- The distributions are right-skewed, with a few heavy-impact categories pulling the tail.